In [ ]:
import numpy as np
import importlib_metadata

print(importlib_metadata.version("numpy"))
if(importlib_metadata.version("numpy") !="1.26.4"):

  !pip uninstall numpy -y
  !pip install numpy==1.26.4
  import os
  os._exit(00)


In [1]:
import os
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

!pip install -U brightway25
import bw2analyzer as ba
import bw2calc as bc
import bw2data as bd
import bw2io as bi

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.2/105.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.6/27.6 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 124.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.8/306.8 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.3/71.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/bw2calc/__init__.py:55: UserWarning: 
It seems like you have an AMD/INTEL x64 architecture, but haven't installed pypardiso:

    https://pypi.org/project/pypardiso/

Installing it could give you much faster calculations.

  warnings.warn(PYPARDISO_WARNING)


In [2]:
def list_files(directory):
    # Iterate over all files in the directory
    for dirpath, _, filenames in os.walk(directory):
        for filename in filenames:
            # Print the full path of each file
            return os.path.join(dirpath, filename)

path_PMF      = "/content/drive/MyDrive/DB/ecoinvent/bw_project/PMF/3.11"
PMF_path_detailed = list_files(path_PMF)
bi.restore_project_directory(PMF_path_detailed,overwrite_existing=True)

path_MI      = "/content/drive/MyDrive/DB/ecoinvent/bw_project/PMF_direct/3.11"
MI_path_detailed = list_files(path_MI)
bi.restore_project_directory(MI_path_detailed,overwrite_existing=True)


Restoring project backup archive - this could take a few minutes...
Restored project: ecoinvent_3.11_PMF
Restoring project backup archive - this could take a few minutes...
Restored project: ecoinvent_3.11_PMF_direct


'ecoinvent_3.11_PMF_direct'

In [3]:
mapping = pd.read_excel("/content/drive/Shareddrives/MIPS ecoinvent method/MI Factors/Vergleich Kassel/MI_PMF_mapping_wbiotic.xlsx",skiprows = 0,sheet_name = "Tabelle1")


In [4]:
bd.projects.set_current("ecoinvent_3.11_PMF")

ei_name = []
for db in list(bd.databases):
  if "cutoff" in db:
    ei_name = db

for m in bd.methods:
  if "PMF Abiotic RMI" == m[0]:
    pmf_abiotic_rmi_method_key = m
  if "PMF Abiotic TMR" == m[0]:
    pmf_abiotic_tmr_method_key = m
  if "PMF Biotic RMI" == m[0]:
    pmf_biotic_rmi_method_key = m
  if "PMF Biotic TMR" == m[0]:
    pmf_biotic_tmr_method_key = m

PMF_abiotic_rmi_vals = []
PMF_abiotic_tmr_vals = []
PMF_biotic_rmi_vals  = []
PMF_biotic_tmr_vals  = []

for i in range(0,len(mapping["ecoinvent_process"])):
  name              = mapping["ecoinvent_process"][i].split("|")[0]
  reference_product = mapping["ecoinvent_process"][i].split("|")[1]
  location          = mapping["location"][i]

  wanted_act = [act for act in bd.Database(ei_name) if name.strip() in act['name']
         and reference_product.strip() in act['reference product']
         and location in act['location']]
  if wanted_act[0]["unit"] != "kilogram":
    print(wanted_act[0]["name"] + "  wrong unit:  " + wanted_act[0]["unit"] )

print("----------------------------")

for i in range(0,len(mapping["ecoinvent_process"])):

  name              = mapping["ecoinvent_process"][i].split("|")[0]
  reference_product = mapping["ecoinvent_process"][i].split("|")[1]
  location          = mapping["location"][i]

  wanted_act = [act for act in bd.Database(ei_name) if name.strip() in act['name']
         and reference_product.strip() in act['reference product']
         and location in act['location']]

  if len(wanted_act)==0:
    print("nothing found for: " + name)
    continue

  if len(wanted_act)>0:
    wanted_act = wanted_act[0]

  print(wanted_act)
  fu = {wanted_act:1}

  lca = bc.LCA(fu, pmf_abiotic_rmi_method_key)
  lca.lci()
  lca.lcia()
  PMF_abiotic_rmi_vals.append(lca.score)

  lca = bc.LCA(fu, pmf_abiotic_tmr_method_key)
  lca.lci()
  lca.lcia()
  PMF_abiotic_tmr_vals.append(lca.score)

  lca = bc.LCA(fu, pmf_biotic_rmi_method_key)
  lca.lci()
  lca.lcia()
  PMF_biotic_rmi_vals.append(lca.score)

  lca = bc.LCA(fu, pmf_biotic_tmr_method_key)
  lca.lci()
  lca.lcia()
  PMF_biotic_tmr_vals.append(lca.score)




market for natural gas, high pressure  wrong unit:  cubic meter
structural timber production  wrong unit:  cubic meter
glued laminated timber production, average glue mix  wrong unit:  cubic meter
market for electricity, low voltage  wrong unit:  kilowatt hour
computer production, laptop  wrong unit:  unit
building construction, multi-storey  wrong unit:  cubic meter
----------------------------
'aluminium production, primary, ingot' (kilogram, IAI Area, EU27 & EFTA, None)
'copper production, cathode, solvent extraction and electrowinning process' (kilogram, GLO, None)
'steel production, low-alloyed, hot rolled' (kilogram, RER, None)
'silver-gold mine operation with refinery' (kilogram, RoW, None)
'primary lead production from concentrate' (kilogram, GLO, None)
'smelting and refining of nickel concentrate, 16% Ni' (kilogram, GLO, None)
'platinum group metal mine operation, ore with high palladium content' (kilogram, RU, None)
'silver-gold mine operation with refinery' (kilogram, RoW, N

KeyboardInterrupt: 

In [ ]:
#method = bd.Method(('PMF Abiotic TMR', 'imaginaryendpoint', 'imaginarymidpoint'))
#method.load()
#method = bd.Method(('PMF Abiotic RMI', 'imaginaryendpoint', 'imaginarymidpoint'))
#method.load()

#method = bd.Method(('PMF Biotic RMI', 'imaginaryendpoint', 'imaginarymidpoint'))
#method.load()

In [17]:
bd.projects.set_current("ecoinvent_3.11_PMF_direct")

ei_name = []
for db in list(bd.databases):
  if "cutoff" in db:
    ei_name = db

for m in bd.methods:
  if "PMF (direct) Abiotic RMI" == m[0]:
    mi_abiotic_rmi_method_key = m
  if "PMF (direct) Abiotic TMR" == m[0]:
    mi_abiotic_tmr_method_key = m
  if "PMF (direct) Biotic RMI" == m[0]:
    mi_biotic_rmi_method_key = m
  if "PMF (direct) Biotic TMR" == m[0]:
    mi_biotic_tmr_method_key = m

product_list = []
process_list = []
location_list = []

MI_abiotic_rmi_vals = []
MI_abiotic_tmr_vals = []
MI_biotic_rmi_vals  = []
MI_biotic_tmr_vals  = []

for i in range(0,len(mapping["ecoinvent_process"])):
  name              = mapping["ecoinvent_process"][i].split("|")[0]
  reference_product = mapping["ecoinvent_process"][i].split("|")[1]
  location          = mapping["location"][i]
  wanted_act = [act for act in bd.Database(ei_name) if name.strip() in act['name']
         and reference_product.strip() in act['reference product']
         and location in act['location']]
  if wanted_act[0]["unit"] != "kilogram":
    print(wanted_act[0]["name"] + "  wrong unit:  " + wanted_act[0]["unit"] )

print("----------------------------")

for i in range(0,len(mapping["ecoinvent_process"])):

  name              = mapping["ecoinvent_process"][i].split("|")[0]
  reference_product = mapping["ecoinvent_process"][i].split("|")[1]
  location          = mapping["location"][i]
  wanted_act = [act for act in bd.Database(ei_name) if name.strip() in act['name']
         and reference_product.strip() in act['reference product']
         and location in act['location']]

  if len(wanted_act)==0:
    print("nothing found for: " + name)
    continue

  if len(wanted_act)>0:
    wanted_act = wanted_act[0]

  product_list.append(mapping["product"][i])
  process_list.append(wanted_act["name"])
  location_list.append(wanted_act["location"])

  print(wanted_act)
  fu = {wanted_act:1}


  lca = bc.LCA(fu, mi_abiotic_rmi_method_key)
  lca.lci()
  lca.lcia()
  MI_abiotic_rmi_vals.append(lca.score)

  lca = bc.LCA(fu, mi_abiotic_tmr_method_key)
  lca.lci()
  lca.lcia()
  MI_abiotic_tmr_vals.append(lca.score)

  lca = bc.LCA(fu, mi_biotic_rmi_method_key)
  lca.lci()
  lca.lcia()
  MI_biotic_rmi_vals.append(lca.score)

  lca = bc.LCA(fu, mi_biotic_tmr_method_key)
  lca.lci()
  lca.lcia()
  MI_biotic_tmr_vals.append(lca.score)


In [ ]:
special = "gold"

bd.projects.set_current("ecoinvent_3.11_PMF")


special_list_raw = bd.Database(ei_name).search(special)
special_list = []

for g in special_list_raw:
  if special in g['reference product']:
    for ex in g.exchanges():
      if "Gangue" in ex["name"]:
        special_list.append(g)

special_PMF_abiotic_rmi_vals = []

for wanted_act in special_list:

  fu = {wanted_act:1}

  lca = bc.LCA(fu, pmf_abiotic_rmi_method_key)
  lca.lci()
  lca.lcia()
  special_PMF_abiotic_rmi_vals.append(lca.score)



bd.projects.set_current("ecoinvent_3.11_PMF_direct")

special_list_raw = bd.Database(ei_name).search(special)
special_list = []

for g in special_list_raw:
  if special in g['reference product']:
    for ex in g.exchanges():
      if "Gangue" in ex["name"]:
        special_list.append(g)

special_MI_abiotic_rmi_vals = []

for wanted_act in special_list:

  fu = {wanted_act:1}

  lca = bc.LCA(fu, mi_abiotic_rmi_method_key)
  lca.lci()
  lca.lcia()
  special_MI_abiotic_rmi_vals.append(lca.score)

special_PMF_abiotic_rmi_vals

pd.DataFrame({"PMF":special_MI_abiotic_rmi_vals,
              "MI":special_PMF_abiotic_rmi_vals})


In [ ]:
df = pd.DataFrame({"product":product_list,
                  "ecoinvent process": process_list,
                  "location": location_list,
                  "PMF Abiotic RMI": PMF_abiotic_rmi_vals,
                  "PMF Abiotic TMR": PMF_abiotic_tmr_vals,
                  "PMF Biotic RMI": PMF_biotic_rmi_vals,
                  "PMF Biotic TMR": PMF_biotic_tmr_vals,
                  "MI Abiotic RMI": MI_abiotic_rmi_vals,
                  "MI Abiotic TMR": MI_abiotic_tmr_vals,
                  "MI Biotic RMI": MI_biotic_rmi_vals,
                  "MI Biotic TMR": MI_biotic_tmr_vals})


df.to_csv("/content/drive/Shareddrives/MIPS ecoinvent method/MI Factors/Vergleich Kassel/results/MI_und_PMF_Faktoren.csv", index = False)

special_processes = []
special_location  = []

for g in special_list:
  special_processes.append(g["name"])
  special_location.append(g["location"])

df_special = pd.DataFrame({"process":special_processes,
                        "location":special_location,
                        "PMF": special_PMF_abiotic_rmi_vals,
                        "MI": special_MI_abiotic_rmi_vals})

df_special.to_csv("/content/drive/Shareddrives/MIPS ecoinvent method/MI Factors/Vergleich Kassel/results/MI_und_PMF_special.csv", index = False)


In [30]:

def print_recursive_calculation(activity, lcia_method, lca_obj=None, total_score=None, amount=1, level=0, max_level=3, cutoff=1e-2,rows=None):
    activity_id = activity.id

    if rows is None:
      rows = []

    if lca_obj is None:
        lca_obj = bc.LCA({activity_id: amount}, lcia_method)
        lca_obj.lci()
        lca_obj.lcia()
        total_score = lca_obj.score
    elif total_score is None:
        raise ValueError
    else:
        lca_obj.redo_lcia({activity_id: amount})
        if abs(lca_obj.score) <= abs(total_score * cutoff):
            return


    print("{}{:4.3f} ({:06.4f}): {:.70}".format("  " * level, lca_obj.score / total_score, lca_obj.score, str(activity_id)))

    # Store result

    rows.append({
        "activity_name": activity["name"],
        "reference_product": activity["reference product"],
        "location": activity["location"],
        "level": level,
        "amount": amount,
        "score": lca_obj.score,
        "relative_contribution": lca_obj.score / total_score if total_score else None,
    })

    if level < max_level:
      for exc in activity.technosphere():
        print_recursive_calculation(
                activity=exc.input,
                lcia_method=lcia_method,
                lca_obj=lca_obj,
                total_score=total_score,
                amount=amount * exc['amount'],
                level=level + 1,
                max_level=max_level,
                cutoff=cutoff,
                rows = rows
            )
    return pd.DataFrame(rows)


laptop   = [act for act in bd.Database(ei_name) if "computer production, laptop" == act['name'] and "GLO" == act['location']][0]
building = [act for act in bd.Database(ei_name) if "building construction, budget hotel" == act['name'] and "BR" == act['location']][0]

bd.projects.set_current("ecoinvent_3.11_PMF")
laptop_tree_pmf_rmi   = print_recursive_calculation(activity = laptop,lcia_method=pmf_abiotic_rmi_method_key,max_level=1)
laptop_tree_pmf_tmr   = print_recursive_calculation(activity = laptop,lcia_method=pmf_abiotic_tmr_method_key,max_level=1)
building_tree_pmf_rmi = print_recursive_calculation(activity = building,lcia_method=pmf_abiotic_rmi_method_key,max_level=1)
building_tree_pmf_tmr = print_recursive_calculation(activity = building,lcia_method=pmf_abiotic_tmr_method_key,max_level=1)


bd.projects.set_current("ecoinvent_3.11_PMF_direct")
laptop_tree_mi_rmi   = print_recursive_calculation(activity = laptop,lcia_method=mi_abiotic_rmi_method_key,max_level=1)
laptop_tree_mi_tmr   = print_recursive_calculation(activity = laptop,lcia_method=mi_abiotic_tmr_method_key,max_level=1)
building_tree_mi_rmi = print_recursive_calculation(activity = building,lcia_method=mi_abiotic_rmi_method_key,max_level=1)
building_tree_mi_tmr = print_recursive_calculation(activity = building,lcia_method=mi_abiotic_tmr_method_key,max_level=1)


laptop_tree_pmf_rmi.to_csv("/content/drive/Shareddrives/MIPS ecoinvent method/MI Factors/Vergleich Kassel/results/laptop_tree_pmf_rmi.csv", index = False)
laptop_tree_pmf_tmr.to_csv("/content/drive/Shareddrives/MIPS ecoinvent method/MI Factors/Vergleich Kassel/results/laptop_tree_pmf_tmr.csv", index = False)
building_tree_pmf_rmi.to_csv("/content/drive/Shareddrives/MIPS ecoinvent method/MI Factors/Vergleich Kassel/results/building_tree_pmf_rmi.csv", index = False)
building_tree_pmf_tmr.to_csv("/content/drive/Shareddrives/MIPS ecoinvent method/MI Factors/Vergleich Kassel/results/building_tree_pmf_tmr.csv", index = False)

laptop_tree_mi_rmi.to_csv("/content/drive/Shareddrives/MIPS ecoinvent method/MI Factors/Vergleich Kassel/results/laptop_tree_mi_rmi.csv", index = False)
laptop_tree_mi_tmr.to_csv("/content/drive/Shareddrives/MIPS ecoinvent method/MI Factors/Vergleich Kassel/results/laptop_tree_mi_tmr.csv", index = False)
building_tree_mi_rmi.to_csv("/content/drive/Shareddrives/MIPS ecoinvent method/MI Factors/Vergleich Kassel/results/building_tree_mi_rmi.csv", index = False)
building_tree_mi_tmr.to_csv("/content/drive/Shareddrives/MIPS ecoinvent method/MI Factors/Vergleich Kassel/results/building_tree_mi_tmr.csv", index = False)





In [ ]:
"""
gold_list = bd.Database(ei_name).search("gold")

goldgangue=[]

for g in gold_list:
  if "gold" in g['reference product']:
    print(g["name"])
    print(g['reference product'])
    for ex in g.exchanges():
      if "Gangue" in ex["name"]:
        goldgangue.append(ex["amount"])
"""

In [ ]:
"""
name              = "platinum group metal mine operation, ore with high palladium content"
reference_product = "platinum"
location          = "RU"
wanted_act = [act for act in bd.Database(ei_name) if name.strip() in act['name']
         and reference_product.strip() in act['reference product']
         and location in act['location']]

for ex in wanted_act[0].exchanges():
  if "Gangue" in ex["name"]:
    print(ex)


method = bd.Method(('MI abiotic TMR', 'imaginaryendpoint', 'imaginarymidpoint'))
method.load()


for db in list(bd.databases):
  if "biosphere" in db:
    biosphere_name = db

for x in bd.Database(biosphere_name):
  if x['categories'][0] == 'natural resource' and x['categories'][1] == 'in ground' and x["unit"] == "standard cubic meter":
    print(x)

bd.Database(biosphere_name).search("Metamorphous rock")

a = bd.Database(ei_name).search("graphite production")[0]
for ex in a.exchanges():
  print(ex)
"""